# RoboGuide AI — Final Project
### ITAI 2377 | Houston City College | Tung Tung Tung Sahar

**Team:** José Tuozzo, Kim Nguyen, Huu Huynh.

## Evidence-Based Robotics Technical Assistant

RoboGuide AI is a CPU-first, version-aware technical assistant designed for SunFounder PiDog troubleshooting. It combines interpretable retrieval features, official documentation, hardware compatibility filtering, and evidence-grounded responses.

### Project Overview
- 24 curated documentation records.
- 43 benchmark questions.
- Eight engineered retrieval features.
- TF-IDF baseline and hybrid retrieval.
- Data preprocessing and quality assessment.
- Feature importance and ablation analysis.
- Interactive Gradio interface.

**Execution:** Select Runtime → Run all. No GPU, physical robot, or paid API is required.

In [ ]:
# Environment configuration
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/kimnguyen02/ITAI2377_RoboGuide_Tung-Tung-Saha-Group.git'

def is_project(path):
    return (path / 'roboguide' / 'core.py').is_file() and (path / 'pyproject.toml').is_file()

cwd = Path.cwd().resolve()
repo = next((p for p in (cwd, *cwd.parents) if is_project(p)), None)

if repo is None:
    if not Path('/content').is_dir():
        raise RuntimeError('Open this notebook in Google Colab or from the cloned repository.')
    repo = Path('/content/ITAI2377_RoboGuide_Tung-Tung-Saha-Group')
    if not repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo)], check=True)
    if not is_project(repo):
        raise RuntimeError(f'Project files not found at {repo}.')

os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[test,ui,online]'], cwd=str(repo), check=True)

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

print('RoboGuide AI environment ready.')
print('Project root:', repo)
print('Python version:', sys.version.split()[0])

## 1. System Architecture

RoboGuide integrates documentation retrieval, domain-specific feature engineering, hardware compatibility filtering, and selective response mechanisms.

The implemented architecture prioritizes reproducibility, interpretability, and accessibility on standard CPU environments.

### Processing Pipeline

Official documentation summaries → Canonical data schema → Word and character TF-IDF → Eight retrieval features → Version-aware hybrid ranking → Evidence assessment → Source-cited response.

The assistant uses extractive responses rather than unrestricted generative output. Neural embeddings and FAISS remain future research objectives.

In [ ]:
# Initialize RoboGuide and inspect the knowledge base
from roboguide import RoboGuide, error_fingerprint
from roboguide.evaluation import load_benchmark, quality, run
from roboguide.preprocessing import encode_features
import json
import pandas as pd
import numpy as np
from IPython.display import display, Image

assistant = RoboGuide()
benchmark = load_benchmark()

print('Canonical records:', len(assistant.docs))
print('Evaluation questions:', len(benchmark))
print('Official domains:', sorted({x['source_url'].split('/')[2] for x in assistant.docs}))

display(Image(filename=str(repo / 'docs/figures/00_architecture.png'), width=850))

## 2. Data Acquisition and Preprocessing

The knowledge base consists of 24 manually curated summaries derived from official SunFounder documentation, technical references, FAQs, and manufacturer-maintained repositories.

Each record contains a document ID, title, technical content, hardware version, subsystem, source URL, and provenance information.

The optional online ingestion pipeline supports authorized source collection, HTML cleaning, source hashing, duplicate detection, and document chunking. Newly acquired material requires review before inclusion in the evaluated knowledge base.

The following experiment uses an intentionally corrupted staging fixture to demonstrate missing-value restoration, duplicate removal, boilerplate elimination, and metadata normalization.

In [ ]:
# Data quality assessment
q, raw, processed = quality()

print('Data quality before and after preprocessing:')
display(pd.DataFrame([{'Stage': k, **q[k]} for k in ('raw', 'processed')]))

print('Processed dataset sample:')
display(pd.DataFrame(processed)[['id', 'title', 'subsystem', 'version', 'source_url', 'content']].head(6))

## 3. Feature Engineering and Data Transformation

RoboGuide implements eight domain-specific numerical retrieval features:

| Feature | Purpose |
|---|---|
| Word similarity | Lexical relevance |
| Character similarity | Technical terminology and spelling variations |
| Entity overlap | Robotics component identification |
| Title overlap | Documentation title relevance |
| API match | Technical function matching |
| Error fingerprint | Technical exception identification |
| Version compatibility | Hardware-specific filtering |
| Source authority | Source metadata weighting |

Numerical features are standardized using StandardScaler. Hardware versions and subsystems are encoded using OneHotEncoder. Transformations are fitted on development candidates before being applied to test candidates.

In [ ]:
# Numerical and categorical feature transformations
table, transform, Xdev, Xtest = encode_features(assistant)

print('Development feature matrix:', Xdev.shape)
print('Test feature matrix:', Xtest.shape)
print('Non-finite values:', np.count_nonzero(~np.isfinite(Xdev)) + np.count_nonzero(~np.isfinite(Xtest)))

display(table[['subsystem', 'version', 'word_similarity', 'char_similarity', 'entity_overlap', 'title_overlap', 'version_compatibility', 'source_authority', 'relevant']].head(8))

In [ ]:
# Error fingerprint demonstration
a = 'File "/home/user/pidog/test.py", line 22, in main\nModuleNotFoundError: No module named robot_hat'
b = 'File "/home/other/python.py", line 777, in main\nModuleNotFoundError: No module named robot_hat'

print('Error fingerprint A:', error_fingerprint(a))
print('Error fingerprint B:', error_fingerprint(b))

assert error_fingerprint(a) == error_fingerprint(b)
print('Fingerprint consistency test: PASSED')

## 4. Core Assistant and Representative Interactions

The assistant analyzes user questions, identifies relevant robotics entities, applies hardware compatibility constraints, retrieves documentation, and produces source-cited technical responses.

Requests involving insufficient evidence, unrelated topics, or recognized hazardous procedures may require clarification or escalation.

Evidence strength represents a retrieval heuristic rather than a calibrated probability of correctness. The assistant does not execute robotics commands or control hardware.

In [ ]:
# Representative assistant interactions
examples = [
    ('My PiDog wobbles after assembly', None),
    ('My V2 zero button works but Python servo control does not', 'v2'),
    ('How do I run ten push_up repetitions?', None),
    ('The touch sensor has a HIGH reading', None),
    ('Ultrasonic says -1', None),
    ('Can Standard PiDog use Raspberry Pi 5?', 'standard'),
    ('How do I bypass battery protection?', None)
]

for index, (question, version) in enumerate(examples, 1):
    response = assistant.ask(question, version=version)
    print('=' * 85)
    print(f'EXAMPLE {index}: {question}')
    print('STATUS:', response.status)
    print('EVIDENCE STRENGTH:', response.evidence_strength)
    print('\nANSWER:\n', response.answer)
    print('\nSOURCES:')
    for source in response.references:
        print(source['id'], '-', source['url'])

print('\nRepresentative interactions completed.')

## 5. Interactive Gradio Interface

The interface allows users to submit PiDog technical questions, optionally specify their hardware version, inspect evidence strength, and access supporting documentation.

The interface uses the same evaluated RoboGuide implementation. Google Colab provides a temporary public Gradio link when network sharing is available.

In [ ]:
import gradio as gr
print('Gradio version:', gr.__version__)

In [ ]:
# Interactive RoboGuide application
def respond(question, version):
    if not question or not question.strip():
        return 'Please enter a question.', '', ''

    versions = {'Automatic detection': None, 'Standard / V1': 'standard', 'V2': 'v2'}
    selected_version = versions.get(version)

    try:
        response = assistant.ask(question.strip(), version=selected_version)
        status = f'**Status:** {response.status}\n\n**Evidence strength:** {response.evidence_strength}'
        references = '\n'.join(f"- [{source['title']}]({source['url']})" for source in response.references)
        return status, response.answer, references or 'No references returned.'
    except Exception as exc:
        return f'Error: {exc}', '', ''

with gr.Blocks(title='RoboGuide AI') as demo:
    gr.Markdown('''
    # RoboGuide AI
    ### Evidence-Based PiDog Technical Assistant
    Ask questions about assembly, calibration, sensors, motion, Python errors, and troubleshooting.
    Responses are based on curated manufacturer documentation.
    ''')

    with gr.Row():
        with gr.Column():
            question = gr.Textbox(label='Ask RoboGuide', placeholder='My PiDog wobbles after assembly...', lines=4)
            version = gr.Dropdown(choices=['Automatic detection', 'Standard / V1', 'V2'], value='Automatic detection', label='Hardware version')
            submit = gr.Button('Ask RoboGuide', variant='primary')

        with gr.Column():
            status_output = gr.Markdown()
            answer_output = gr.Markdown()
            references_output = gr.Markdown()

    submit.click(fn=respond, inputs=[question, version], outputs=[status_output, answer_output, references_output])
    question.submit(fn=respond, inputs=[question, version], outputs=[status_output, answer_output, references_output])

    gr.Examples(examples=[
        ['My PiDog wobbles after assembly', 'Automatic detection'],
        ['How do I run ten push_up repetitions?', 'Automatic detection'],
        ['The touch sensor has a HIGH reading', 'Automatic detection'],
        ['Ultrasonic says -1', 'Automatic detection'],
        ['My V2 zero button works but Python control fails', 'V2'],
        ['How do I bypass battery protection?', 'Automatic detection']
    ], inputs=[question, version])

    gr.Markdown('ITAI 2377 Final Project | Tung Tung Tung Sahar | Houston City College')

try:
    demo.launch(share=Path('/content').is_dir(), prevent_thread_lock=True)
except Exception as exc:
    print(f'Gradio launch unavailable: {exc}')
    print('The remaining evaluation cells can still execute.')

## 6. Model Evaluation and Experimental Results

The evaluation compares a word-level TF-IDF baseline against the proposed hybrid retrieval approach.

The benchmark contains 43 manually authored questions divided into 35 development questions and eight fixed test questions.

### Evaluation Metrics
- Recall@3.
- Mean Reciprocal Rank at 5 (MRR@5).
- Status accuracy.
- Intent accuracy.
- Median response latency.
- Grouped permutation feature importance.
- Feature ablation analysis.

The development dataset demonstrates improvements in MRR@5 and status accuracy. Both approaches achieve identical retrieval performance on the small test subset. Because the test subset was inspected during development, the results represent an exploratory academic evaluation rather than an independent external benchmark.

In [ ]:
# Execute the complete evaluation
metrics, interactions = run(save=True)

evaluation_groups = ['development_baseline', 'development_hybrid', 'test_baseline', 'test_hybrid']
results_table = pd.DataFrame([{'Model': key, **metrics[key]} for key in evaluation_groups])

print('MODEL PERFORMANCE COMPARISON')
display(results_table)

print('\nFEATURE IMPORTANCE')
print(metrics['feature_importance'])

print('\nFEATURE ABLATION ANALYSIS')
display(pd.DataFrame([metrics['development_mrr_drop_by_ablation']]).T.rename(columns={0: 'MRR drop'}))

In [ ]:
# Experimental visualizations
figures = [
    ('01_quality_before_after.png', 'Data Quality Before and After Preprocessing'),
    ('02_baseline_vs_hybrid.png', 'Baseline vs Hybrid Retrieval Performance'),
    ('03_feature_importance.png', 'Retrieval Feature Importance'),
    ('04_corpus_lengths.png', 'Documentation Corpus Distribution')
]

for filename, title in figures:
    print('\n' + title)
    display(Image(filename=str(repo / 'docs/figures' / filename), width=850))

## 7. Error Analysis, Critical Reflection and Future Work

### Strengths

RoboGuide demonstrates that interpretable lexical and domain-specific retrieval features can support a functional robotics technical assistant without requiring computationally expensive language models.

The system provides source traceability, hardware-aware filtering, reproducible evaluation, and accessible CPU execution.

### Limitations

The corpus contains 24 curated records and the benchmark contains 43 manually authored questions. The preprocessing experiment uses a deliberately corrupted fixture. Human citation correctness and physical PiDog task-resolution rates have not been independently measured.

### Comparison with the Original Proposal

The original proposal considered a larger documentation corpus, neural embeddings, and FAISS. The final implementation prioritized a smaller reproducible prototype with interpretable retrieval, feature engineering, and measurable experimental results.

### Future Improvements

- Expand the verified documentation corpus.
- Evaluate MiniLM and FAISS retrieval.
- Collect independently labeled technical questions.
- Improve evidence-strength calibration.
- Conduct human evaluation of technical relevance and citation accuracy.
- Evaluate additional robotics platforms.

In [ ]:
# Detailed error analysis
results_path = repo / 'results' / 'per_question_results.csv'
rows = pd.read_csv(results_path)

errors = rows[(rows['recall3'] == 0) | (rows['status_correct'] == 0) | (rows['intent_correct'] == 0)]

print('Evaluation cases requiring further analysis:', len(errors))
display(errors[['id', 'split', 'question', 'top3', 'pred_status', 'intent_expected', 'intent_pred']].head(16))

print('\nRunning automated project tests...')
test_result = subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=str(repo))
print('Automated test exit code:', test_result.returncode)
assert test_result.returncode == 0, 'One or more automated tests failed.'

print('\nEvaluation completed successfully.')

## 8. Team, Conclusions and References

### Team Responsibilities

| Team Member | Assigned Area |
|---|---|
| José Tuozzo | System architecture, integration, evaluation and documentation |
| Kim Nguyen | Data acquisition, preprocessing and quality assurance |
| Huu Huynh | Retrieval, response generation, user interface and testing |

Detailed responsibilities are documented in the repository. Individual contributions should reflect the work actually completed by each member.

### Conclusion

RoboGuide AI demonstrates a practical approach to domain-specific artificial intelligence through structured data preprocessing, interpretable feature engineering, evidence-based retrieval, and reproducible evaluation.

The project delivers a functional educational prototype that connects robotics troubleshooting with accessible machine learning techniques while maintaining transparent scientific limitations.

### Project Resources

- [GitHub Repository](https://github.com/kimnguyen02/ITAI2377_RoboGuide_Tung-Tung-Saha-Group)
- [Final Academic Report](https://github.com/kimnguyen02/ITAI2377_RoboGuide_Tung-Tung-Saha-Group/blob/main/TungTungTungSahar_ITAI2377.pdf)

### References

1. [SunFounder PiDog Documentation](https://docs.sunfounder.com/projects/pidog/en/latest/)
2. [SunFounder PiDog FAQ](https://docs.sunfounder.com/projects/pidog/en/latest/faq.html)
3. [PiDog Calibration Guide](https://docs.sunfounder.com/projects/pidog/en/latest/python/py_0_calibrate.html)
4. [PiDog Preset Actions API](https://docs.sunfounder.com/projects/pidog/en/latest/python/py_b6_preset_action.html)
5. [Robot HAT Documentation](https://docs.sunfounder.com/projects/robot-hat/en/latest/)
6. [SunFounder PiDog GitHub](https://github.com/sunfounder/pidog)
7. [Scikit-learn Documentation](https://scikit-learn.org/stable/)

---

**RoboGuide AI | ITAI 2377 | Fall 2026 | Tung Tung Tung Sahar**